# M11. transpose (행과 열 뒤집기)

> 📌 **언제 필요한가**  
> 받은 데이터가 "행=항목, 열=연도" 처럼 우리가 원하는 거랑 뒤집혀 있을 때.  
> 인천 인구추이 데이터가 대표적.

## 이 모듈에서 배울 것

- `.T`로 transpose
- transpose 전후 인덱스 처리
- 언제 transpose가 필요한지 판단

---


## 📥 데이터 준비

> 이 모듈은 아래 파일이 필요해요. **저장소에는 동봉돼 있지 않으니** 먼저 받아서 두세요.
> 받는 곳 링크를 누르면 바로 받으러 갈 수 있어요.

- `인천광역시_인구추이_현황_20231231.csv` — 인코딩 `cp949` — [공공데이터포털에서 받기](https://www.data.go.kr/data/15055013/fileData.do)

두는 곳 — **로컬 Jupyter**: 이 노트북과 같은 폴더 / **Colab**: `/content/`에 업로드.  
컬럼 설명·함정 등 자세한 내용은 [`data/README.md`](data/README.md) 참고.

---

## 1. 어, 행/열이 거꾸로네?

인천광역시 인구추이 데이터를 봅시다.


In [ ]:
import pandas as pd

df = pd.read_csv('data/인천광역시_인구추이 현황.csv', encoding='cp949')
print(f"shape: {df.shape}")
df


**눈치채셨어요?** 행이 항목(세대수, 등록인구 등)이고 열이 연도예요.

우리가 원하는 건 보통 반대 — **행이 시점, 열이 항목**.

| | 우리가 원하는 형태 | 받은 형태 |
|---|---|---|
| 행 | 연도 | 항목 |
| 열 | 항목 | 연도 |

→ **transpose (행/열 뒤집기)** 필요.


## 2. 첫 시도 — 그냥 `.T`


In [ ]:
df_T = df.T
df_T


결과가 이상해요. 첫 컬럼('인구추이별')도 transpose된 거에 같이 섞임. 이게 컬럼명이 되어야 하는데.

**해결**: 컬럼을 먼저 인덱스로 설정한 다음 transpose.


## 3. 깔끔한 transpose


In [ ]:
# Step 1: '인구추이별'을 인덱스로 (그래야 transpose 후 컬럼명이 됨)
df_indexed = df.set_index('인구추이별')

# Step 2: 부가 정보 컬럼 제거 (인구추이별(1), 인구추이별(2))
df_indexed = df_indexed.drop(columns=['인구추이별(1)', '인구추이별(2)'])
df_indexed.head()


In [ ]:
# Step 3: transpose
df_clean = df_indexed.T
df_clean.head()


**🎉 이제 행이 연도, 열이 항목이에요!**


## 4. 정리 후 사용


In [ ]:
# 인덱스(연도) 정수로 변환
df_clean.index = df_clean.index.astype(int)
df_clean.index.name = '연도'

# 한국인 인구 (남+여) 계산
df_clean['한국인_인구'] = df_clean['등록인구 (명)남-한국인'] + df_clean['등록인구 (명)여-한국인']

df_clean[['세대수 (세대)', '한국인_인구', '인구밀도 (명/제곱킬로미터)']].head()


## 5. 시각화 — 이제 시계열로 자연스럽게


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(df_clean.index, df_clean['한국인_인구'], 'o-')
plt.xlabel('Year')
plt.ylabel('Korean population')
plt.title('Incheon Korean population')
plt.grid(alpha=0.3)
plt.show()


## 6. 본인 데이터에 적용해보기 ✏️

받은 데이터가 행/열 뒤집혔으면:


In [ ]:
# Step 1: 키 컬럼을 인덱스로
# my_df = my_df.set_index('항목명_컬럼')
# 
# # Step 2: 불필요 컬럼 제거 (있으면)
# my_df = my_df.drop(columns=['필요없는_컬럼'])
# 
# # Step 3: transpose
# my_df_T = my_df.T


## 7. ⚠️ 함정 / 주의사항

### 7.1 transpose는 모든 데이터를 같은 타입으로 강제
숫자 컬럼과 문자 컬럼이 섞여 있으면 transpose 후 다 `object` 타입 됨.  
**해결**: transpose 후 `astype`으로 다시 변환.

### 7.2 컬럼명이 길어짐
원래 행 이름이 컬럼명이 되니까 한글 + 단위로 길어질 수 있음.  
**해결**: M04 모듈의 rename으로 짧게.

### 7.3 인덱스 이름
transpose 후 인덱스 이름이 원래 컬럼명이 됨. 의미 없으면 변경:
```python
df_T.index.name = '연도'
```

### 7.4 인덱스가 문자열인 경우
"2018" 같은 문자열로 들어오면 정렬/그래프가 이상.  
**해결**: `df.index = df.index.astype(int)`


## 8. 📚 더 알아보기

- `pd.melt()` — wide → long 형태로 (transpose보다 유연)
- `pd.pivot()` — long → wide
- `df.stack()` / `df.unstack()` — 멀티인덱스로 차원 이동
